# Active Learning for Object Detection

A structured approach to active learning for optimizing data selection in YOLO-based object detection.

## Overview

This notebook provides a comprehensive four-step workflow for implementing active learning with YOLO models. The goal is to make data-driven decisions for each component of the pipeline based on measurable success metrics.

### Workflow Structure:

1. **Part 1: Feature Extraction & Layer Analysis**  
   Analyze all model layers to find which produces the most separable features for clustering.

2. **Part 2: Clustering & Visualization**  
   Compare K-Means and HDBSCAN, visualize with t-SNE, and analyze data density.

3. **Part 3: Active Learning Selection Strategies**  
   Evaluate multiple strategies (Random, Uncertainty, Cluster-based, Diversity).

4. **Part 4: Training & Evaluation**  
   Train the next iteration with selected data and evaluate performance.

## Requirements

```bash
pip install ultralytics torch numpy pandas matplotlib seaborn scikit-learn tqdm hdbscan
```

---
## ⚙️ Setup and Configuration

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
import os
from ultralytics import YOLO
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
from pathlib import Path
from tqdm.notebook import tqdm
from typing import List, Dict, Tuple, Any

# Attempt to import HDBSCAN
try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except ImportError:
    HDBSCAN_AVAILABLE = False
    print("⚠️ HDBSCAN is not installed and will be unavailable. To install: pip install hdbscan")

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# --- Centralized Configuration ---
class Config:
    """Configuration class for all experiment parameters."""
    
    # --- Paths (UPDATE THESE FOR YOUR SETUP) ---
    MODEL_PATH = "path/to/your/model/weights/best.pt"  # Path to the model for feature extraction
    IMAGE_DIR = "path/to/unlabeled/pool/images"         # Directory of images to analyze
    LABEL_DIR = "path/to/unlabeled/pool/labels"         # Directory of corresponding labels
    CONFIDENCE_FILES_DIR = "runs/obb/val/labels"        # Directory with prediction .txt files (containing confidences)
    RESULTS_DIR = Path("active_learning_results")       # Where to save outputs like plots and data
    CACHE_DIR = RESULTS_DIR / "feature_cache"           # Where to cache extracted features
    DESTINATION_DATASET_FOLDER = "YOLO_Next_Training_Pool"  # Where to save selected images/labels

    # --- Parameters ---
    BATCH_SIZE = 16
    N_IMAGES_TO_SELECT = 200      # Number of images to select for the next pool
    KMEANS_N_CLUSTERS = 30        # Number of clusters for KMeans
    HDBSCAN_MIN_CLUSTER_SIZE = 15
    HDBSCAN_MIN_SAMPLES = 5
    DENSITY_K_NEIGHBORS = 10      # Number of neighbors for density calculation

# Create results and cache directories
Config.RESULTS_DIR.mkdir(exist_ok=True)
Config.CACHE_DIR.mkdir(exist_ok=True)

print(f"Results will be saved to: {Config.RESULTS_DIR}")

In [ ]:
# --- Load Model and Image Paths ---
print("Loading model...")
model = YOLO(Config.MODEL_PATH)

image_paths = sorted([p for p in Path(Config.IMAGE_DIR).glob('*.jpg')])
# Add other extensions if needed
image_paths.extend(sorted([p for p in Path(Config.IMAGE_DIR).glob('*.png')]))
image_paths.extend(sorted([p for p in Path(Config.IMAGE_DIR).glob('*.jpeg')]))

if not image_paths:
    raise FileNotFoundError(f"No images found in {Config.IMAGE_DIR}")
    
print(f"✅ Found {len(image_paths)} images.")

---
## Dataset Filtering Utility

Utility function to filter datasets by removing already-selected images.

In [ ]:
def filter_yolo_dataset(source_dir: Path, selection_dir: Path, output_dir: Path):
    """
    Filters a YOLO dataset by removing images and labels found in a selection set.
    
    Use this between active learning iterations to remove already-selected samples
    from the unlabeled pool.

    Args:
        source_dir (Path): Path to the source dataset directory.
        selection_dir (Path): Path to the dataset with items to remove.
        output_dir (Path): Path where the filtered dataset will be saved.
    """
    # 1. Define and validate paths
    source_images = source_dir / 'images'
    source_labels = source_dir / 'labels'
    selection_labels = selection_dir / 'labels'

    for path in [source_images, source_labels, selection_labels]:
        if not path.is_dir():
            raise FileNotFoundError(f"❌ Required directory not found: {path}")

    # 2. Prepare output directories
    output_images = output_dir / 'images'
    output_labels = output_dir / 'labels'
    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)

    # 3. Get a set of file stems to remove
    stems_to_remove = {p.stem for p in selection_labels.glob('*.txt')}
    print(f"🔎 Found {len(stems_to_remove)} unique file stems to remove.")

    # 4. Iterate through source images, copying only the ones to keep
    source_image_paths = [p for p in source_images.iterdir() if p.is_file()]
    kept_count = 0
    for img_path in source_image_paths:
        if img_path.stem in stems_to_remove:
            continue  # Skip this file

        label_path = source_labels / f"{img_path.stem}.txt"

        # Copy image and its corresponding label, if it exists
        shutil.copy2(img_path, output_images)
        if label_path.exists():
            shutil.copy2(label_path, output_labels)
        
        kept_count += 1
    
    # 5. Copy configuration file, if present
    source_yaml = source_dir / 'data.yaml'
    if source_yaml.exists():
        shutil.copy2(source_yaml, output_dir)

    # 6. Print a final summary
    total_count = len(source_image_paths)
    print("\n--- Filtering Complete ---")
    print(f"Total source images: {total_count}")
    print(f"✅ Images/labels kept: {kept_count}")
    print(f"🗑️ Images/labels removed: {total_count - kept_count}")
    print(f"🎉 Filtered dataset ready at: {output_dir}")

In [ ]:
# Example: Filter dataset after iteration
# filter_yolo_dataset(
#     source_dir=Path('datasets/unlabeled_pool'),
#     selection_dir=Path('datasets/iteration_1_selected'),
#     output_dir=Path('datasets/unlabeled_pool_filtered')
# )

---
## Part 1: Find the Best Layer for Feature Extraction

**Objective:** Identify the model layer that produces the most structured and separable feature vectors, which is crucial for effective clustering.

**Methodology:**
1. Iterate through each layer of the YOLO model.
2. Extract feature vectors for all images. **A caching system is used to avoid re-extracting on subsequent runs.**
3. Evaluate feature quality by clustering with K-Means and HDBSCAN.
4. Use **Silhouette** and **Calinski-Harabasz** scores to measure success. For both metrics, higher scores are better.

In [ ]:
def extract_features_from_layer(model, image_paths: List[Path], layer_idx: int, 
                                 batch_size: int = 16) -> np.ndarray:
    """
    Extract features from a specific layer of the YOLO model.
    
    Args:
        model: YOLO model instance.
        image_paths: List of paths to images.
        layer_idx: Index of the layer to extract features from.
        batch_size: Batch size for processing.
        
    Returns:
        NumPy array of feature vectors.
    """
    features_list = []
    
    for i in tqdm(range(0, len(image_paths), batch_size), desc=f"Layer {layer_idx}"):
        batch_paths = image_paths[i:i + batch_size]
        batch_images = [str(p) for p in batch_paths]
        
        # Run inference with embed parameter to get intermediate features
        results = model.predict(batch_images, embed=[layer_idx], verbose=False)
        
        for result in results:
            if hasattr(result, 'embed') and result.embed is not None:
                # Global average pooling to get fixed-size feature vector
                feat = result.embed[0]
                if len(feat.shape) > 1:
                    feat = feat.mean(dim=(1, 2)) if len(feat.shape) == 3 else feat.mean(dim=0)
                features_list.append(feat.cpu().numpy())
    
    return np.array(features_list)


def evaluate_clustering(features: np.ndarray, n_clusters: int = 30) -> Dict[str, float]:
    """
    Evaluate feature quality using clustering metrics.
    
    Args:
        features: Feature matrix.
        n_clusters: Number of clusters for K-Means.
        
    Returns:
        Dictionary with clustering scores.
    """
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    results = {}
    
    # K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans_labels = kmeans.fit_predict(features_scaled)
    
    results['kmeans_silhouette'] = silhouette_score(features_scaled, kmeans_labels)
    results['kmeans_calinski'] = calinski_harabasz_score(features_scaled, kmeans_labels)
    
    # HDBSCAN
    if HDBSCAN_AVAILABLE:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=Config.HDBSCAN_MIN_CLUSTER_SIZE,
            min_samples=Config.HDBSCAN_MIN_SAMPLES
        )
        hdbscan_labels = clusterer.fit_predict(features_scaled)
        
        # Only calculate scores if we have valid clusters
        valid_mask = hdbscan_labels != -1
        if valid_mask.sum() > 1 and len(np.unique(hdbscan_labels[valid_mask])) > 1:
            results['hdbscan_silhouette'] = silhouette_score(
                features_scaled[valid_mask], hdbscan_labels[valid_mask]
            )
            results['hdbscan_calinski'] = calinski_harabasz_score(
                features_scaled[valid_mask], hdbscan_labels[valid_mask]
            )
        else:
            results['hdbscan_silhouette'] = 0.0
            results['hdbscan_calinski'] = 0.0
    
    return results

In [ ]:
def analyze_all_layers(model, image_paths: List[Path], cache_dir: Path) -> pd.DataFrame:
    """
    Analyze all layers to find the best one for feature extraction.
    
    Args:
        model: YOLO model instance.
        image_paths: List of paths to images.
        cache_dir: Directory to cache features.
        
    Returns:
        DataFrame with analysis results.
    """
    # Get number of layers from model
    n_layers = len(model.model.model)
    print(f"--- Starting Layer Analysis for {n_layers} layers ---")
    
    results = []
    
    for layer_idx in tqdm(range(n_layers), desc="Analyzing All Layers"):
        cache_file = cache_dir / f"features_layer_{layer_idx}.npy"
        
        # Load from cache or extract
        if cache_file.exists():
            features = np.load(cache_file)
        else:
            try:
                features = extract_features_from_layer(model, image_paths, layer_idx)
                np.save(cache_file, features)
            except Exception as e:
                print(f"Skipping layer {layer_idx}: {e}")
                continue
        
        if len(features) == 0 or features.ndim != 2:
            continue
            
        # Evaluate clustering
        scores = evaluate_clustering(features, Config.KMEANS_N_CLUSTERS)
        scores['layer_index'] = layer_idx
        scores['feature_dim'] = features.shape[1]
        results.append(scores)
    
    return pd.DataFrame(results)

In [ ]:
# Run layer analysis (uncomment to execute)
# layer_results_df = analyze_all_layers(model, image_paths, Config.CACHE_DIR)
# layer_results_df.to_csv(Config.RESULTS_DIR / 'layer_analysis_results.csv', index=False)
# print(layer_results_df)

In [ ]:
def plot_layer_analysis(df: pd.DataFrame, save_path: Path):
    """Plot layer analysis results."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Layer Performance by Clustering Score (Higher is Better)', fontsize=14, weight='bold')

    # Silhouette Scores
    ax1 = axes[0]
    x = df['layer_index']
    ax1.bar(x - 0.2, df['kmeans_silhouette'], width=0.4, label='K-Means', color='skyblue')
    if 'hdbscan_silhouette' in df.columns:
        ax1.bar(x + 0.2, df['hdbscan_silhouette'], width=0.4, label='HDBSCAN', color='salmon')
    ax1.set_xlabel('Layer Index')
    ax1.set_ylabel('Silhouette Score')
    ax1.set_title('Silhouette Score by Layer')
    ax1.legend()
    
    # Best layer marker
    best_idx = df['kmeans_silhouette'].idxmax()
    ax1.axvline(x=df.loc[best_idx, 'layer_index'], color='red', linestyle='--', 
                label=f'Best: Layer {df.loc[best_idx, "layer_index"]}')

    # Calinski-Harabasz Scores
    ax2 = axes[1]
    ax2.bar(x - 0.2, df['kmeans_calinski'], width=0.4, label='K-Means', color='skyblue')
    if 'hdbscan_calinski' in df.columns:
        ax2.bar(x + 0.2, df['hdbscan_calinski'], width=0.4, label='HDBSCAN', color='salmon')
    ax2.set_xlabel('Layer Index')
    ax2.set_ylabel('Calinski-Harabasz Score')
    ax2.set_title('Calinski-Harabasz Score by Layer')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    
    print(f"\n🏆 Best layer for Silhouette: {df.loc[best_idx, 'layer_index']}")


# Plot results (uncomment after running analysis)
# plot_layer_analysis(layer_results_df, Config.RESULTS_DIR / 'layer_performance.png')

---
## Part 2: Clustering Visualization and Density Analysis

**Objective:** Visualize the clustering quality and analyze feature space density.

**Methodology:**
1. Extract features from the best layer identified in Part 1.
2. Apply both K-Means and HDBSCAN.
3. Use **t-SNE** to reduce dimensions to 2D for visualization.
4. Calculate and visualize density scores.

In [ ]:
# Configuration for Part 2
BEST_LAYER_IDX = 2  # UPDATE THIS based on Part 1 results

In [ ]:
def perform_clustering_analysis(features: np.ndarray, n_clusters: int = 30) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Perform clustering and density analysis.
    
    Returns:
        Tuple of (scaled_features, cluster_labels, centroids)
    """
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    # K-Means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(features_scaled)
    centroids = kmeans.cluster_centers_
    
    return features_scaled, labels, centroids


def compute_density_scores(features: np.ndarray, k: int = 10) -> np.ndarray:
    """
    Compute density scores using k-nearest neighbors.
    Lower scores indicate sparser (more informative) regions.
    """
    nn = NearestNeighbors(n_neighbors=k + 1)  # +1 because point is its own neighbor
    nn.fit(features)
    distances, _ = nn.kneighbors(features)
    
    # Average distance to k neighbors (excluding self)
    density_scores = distances[:, 1:].mean(axis=1)
    return density_scores


def visualize_clusters_tsne(features: np.ndarray, labels: np.ndarray, 
                            density_scores: np.ndarray, save_path: Path):
    """
    Visualize clusters using t-SNE.
    """
    print("Running t-SNE dimensionality reduction...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    features_2d = tsne.fit_transform(features)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle('Feature Space Visualization', fontsize=14, weight='bold')
    
    # Cluster visualization
    scatter1 = axes[0].scatter(features_2d[:, 0], features_2d[:, 1], 
                                c=labels, cmap='tab20', alpha=0.6, s=10)
    axes[0].set_title('Cluster Assignments')
    axes[0].set_xlabel('t-SNE 1')
    axes[0].set_ylabel('t-SNE 2')
    plt.colorbar(scatter1, ax=axes[0], label='Cluster')
    
    # Density visualization
    scatter2 = axes[1].scatter(features_2d[:, 0], features_2d[:, 1], 
                                c=density_scores, cmap='viridis', alpha=0.6, s=10)
    axes[1].set_title('Density Scores (Higher = Sparser Region)')
    axes[1].set_xlabel('t-SNE 1')
    axes[1].set_ylabel('t-SNE 2')
    plt.colorbar(scatter2, ax=axes[1], label='Density Score')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    
    return features_2d

In [ ]:
# Run clustering analysis (uncomment to execute)
# print(f"--- Part 2: Analyzing Best Layer ({BEST_LAYER_IDX}) ---")

# # Load or extract features
# cache_file = Config.CACHE_DIR / f"features_layer_{BEST_LAYER_IDX}.npy"
# if cache_file.exists():
#     features = np.load(cache_file)
# else:
#     features = extract_features_from_layer(model, image_paths, BEST_LAYER_IDX)
#     np.save(cache_file, features)

# # Perform analysis
# features_scaled, final_labels, final_centroids = perform_clustering_analysis(
#     features, Config.KMEANS_N_CLUSTERS
# )
# density_scores = compute_density_scores(features_scaled, Config.DENSITY_K_NEIGHBORS)

# # Visualize
# features_2d = visualize_clusters_tsne(
#     features_scaled, final_labels, density_scores,
#     Config.RESULTS_DIR / 'clustering_visualization.png'
# )

---
## Part 3: Active Learning Selection Strategies

**Objective:** Evaluate and apply different active learning selection strategies.

### Strategies Implemented:
1. **Random** - Baseline random sampling
2. **Pure Uncertainty** - Select samples with lowest model confidence
3. **Cluster-Proportional Uncertainty** - Select uncertain samples proportionally from each cluster
4. **Typical Diversity** - Select samples closest to cluster centroids (representative samples)

In [ ]:
def load_confidence_scores(image_paths: List[Path], confidence_dir: str) -> Tuple[List[Path], np.ndarray]:
    """
    Load confidence scores from prediction files.
    
    Returns:
        Tuple of (filtered_paths, confidence_scores)
    """
    confidence_path = Path(confidence_dir)
    if not confidence_path.is_dir():
        raise FileNotFoundError(f"Confidence directory not found: {confidence_path}")
    
    valid_paths = []
    confidences = []
    
    for img_path in tqdm(image_paths, desc="Loading confidences"):
        conf_file = confidence_path / f"{img_path.stem}.txt"
        if conf_file.exists() and conf_file.stat().st_size > 0:
            try:
                score_text = conf_file.read_text().strip().split()[0]
                confidence = float(score_text)
                valid_paths.append(img_path)
                confidences.append(confidence)
            except (ValueError, IndexError):
                continue
    
    print(f"✅ Loaded {len(valid_paths)} confidence scores.")
    return valid_paths, np.array(confidences)

In [ ]:
def apply_selection_strategies(
    features_scaled: np.ndarray,
    labels: np.ndarray,
    centroids: np.ndarray,
    confidences: np.ndarray,
    n_select: int
) -> Dict[str, np.ndarray]:
    """
    Apply all selection strategies and return selected indices.
    
    Returns:
        Dictionary mapping strategy names to selected indices.
    """
    selection_results = {}
    indices_range = np.arange(len(features_scaled))
    
    # A: Random Baseline
    np.random.seed(42)
    selection_results['Random'] = np.random.choice(indices_range, size=n_select, replace=False)
    
    # B: Pure Uncertainty (lowest confidence)
    selection_results['Pure Uncertainty'] = np.argsort(confidences)[:n_select]
    
    # C: Cluster-Proportional Uncertainty
    indices_c = []
    cluster_ids, counts = np.unique(labels[labels != -1], return_counts=True)
    if len(cluster_ids) > 0:
        proportions = counts / counts.sum()
        for i, cluster_id in enumerate(cluster_ids):
            n_from_cluster = max(1, int(np.round(proportions[i] * n_select)))
            in_cluster_mask = np.where(labels == cluster_id)[0]
            if len(in_cluster_mask) > 0:
                cluster_confs = confidences[in_cluster_mask]
                sorted_indices = in_cluster_mask[np.argsort(cluster_confs)]
                indices_c.extend(sorted_indices[:n_from_cluster])
    
    indices_c = list(dict.fromkeys(indices_c))  # Remove duplicates, preserve order
    if len(indices_c) < n_select:
        additional = [idx for idx in selection_results['Pure Uncertainty'] if idx not in indices_c]
        indices_c.extend(additional[:n_select - len(indices_c)])
    selection_results['Cluster Uncertainty'] = np.array(indices_c[:n_select])
    
    # D: Typical Diversity (samples closest to centroids)
    indices_d = []
    unique_labels = np.unique(labels[labels != -1])
    if len(centroids) > 0 and len(unique_labels) > 0:
        samples_per_cluster = max(1, n_select // len(unique_labels))
        for i, cluster_id in enumerate(unique_labels):
            if i >= len(centroids):
                break
            in_cluster_mask = np.where(labels == cluster_id)[0]
            if len(in_cluster_mask) > 0:
                cluster_features = features_scaled[in_cluster_mask]
                distances = cdist(cluster_features, centroids[i].reshape(1, -1)).flatten()
                most_typical_indices = in_cluster_mask[np.argsort(distances)]
                indices_d.extend(most_typical_indices[:samples_per_cluster])
    
    indices_d = list(dict.fromkeys(indices_d))
    if len(indices_d) < n_select:
        remaining = [idx for idx in indices_range if idx not in indices_d]
        indices_d.extend(np.random.choice(remaining, size=min(n_select - len(indices_d), len(remaining)), replace=False))
    selection_results['Typical Diversity'] = np.array(indices_d[:n_select])
    
    return selection_results

In [ ]:
def visualize_strategy_comparison(
    selection_results: Dict[str, np.ndarray],
    confidences: np.ndarray,
    save_path: Path
):
    """
    Visualize and compare selection strategies.
    """
    # Calculate metrics
    results_data = []
    for name, indices in selection_results.items():
        results_data.append({
            'Strategy': name,
            'Mean Confidence': np.mean(confidences[indices]),
            'Std Confidence': np.std(confidences[indices]),
            'Min Confidence': np.min(confidences[indices]),
            'Max Confidence': np.max(confidences[indices])
        })
    
    results_df = pd.DataFrame(results_data).sort_values('Mean Confidence').reset_index(drop=True)
    
    print("\n📊 Strategy Comparison (Lower mean confidence = more uncertain samples):")
    print(results_df.to_string(index=False))
    
    # Visualization
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Active Learning Strategy Comparison', fontsize=14, weight='bold')
    
    # Bar plot
    sns.barplot(x='Mean Confidence', y='Strategy', 
                data=results_df.sort_values('Mean Confidence', ascending=False),
                ax=axes[0], palette='viridis')
    axes[0].set_title('Strategy Performance (Lower is Better)')
    axes[0].axvline(x=confidences.mean(), color='r', ls='--', 
                    label=f'Dataset Mean ({confidences.mean():.3f})')
    axes[0].legend()
    
    # Distribution plot
    sns.kdeplot(confidences, ax=axes[1], fill=True, color='grey', label='Entire Pool')
    top_strategies = results_df['Strategy'].head(2).tolist()
    colors = ['blue', 'orange']
    for i, strategy in enumerate(top_strategies):
        sns.kdeplot(confidences[selection_results[strategy]], ax=axes[1], 
                   fill=True, alpha=0.5, label=strategy, color=colors[i])
    axes[1].set_title('Distribution of Selected Confidences')
    axes[1].set_xlabel('Confidence Score')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    
    return results_df

In [ ]:
# Run selection strategies (uncomment to execute)
# print("\n--- Part 3: Applying Selection Strategies ---")

# # Load confidence scores
# valid_paths, confidences = load_confidence_scores(image_paths, Config.CONFIDENCE_FILES_DIR)

# # Apply strategies
# selection_results = apply_selection_strategies(
#     features_scaled, final_labels, final_centroids, 
#     confidences, Config.N_IMAGES_TO_SELECT
# )

# # Visualize
# strategy_df = visualize_strategy_comparison(
#     selection_results, confidences,
#     Config.RESULTS_DIR / 'strategy_comparison.png'
# )

In [ ]:
def export_selected_dataset(
    image_paths: List[Path],
    selected_indices: np.ndarray,
    label_dir: Path,
    output_dir: Path
):
    """
    Export selected images and labels to a new directory.
    """
    output_images = output_dir / 'images'
    output_labels = output_dir / 'labels'
    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)
    
    copied = 0
    for idx in tqdm(selected_indices, desc="Exporting selected data"):
        img_path = image_paths[idx]
        label_path = label_dir / f"{img_path.stem}.txt"
        
        shutil.copy2(img_path, output_images / img_path.name)
        if label_path.exists():
            shutil.copy2(label_path, output_labels / label_path.name)
        copied += 1
    
    print(f"\n✅ Exported {copied} images/labels to: {output_dir}")

In [ ]:
# Export selected dataset (uncomment to execute)
# CHOSEN_STRATEGY = 'Typical Diversity'  # Choose your preferred strategy

# export_selected_dataset(
#     valid_paths,
#     selection_results[CHOSEN_STRATEGY],
#     Path(Config.LABEL_DIR),
#     Path(Config.DESTINATION_DATASET_FOLDER)
# )

---
## Part 4: Training and Evaluation

Train the model with the newly selected data and evaluate performance.

In [ ]:
def train_model(model_config: str, pretrained: str, data_yaml: str, 
                epochs: int = 300, imgsz: int = 1024, batch: int = 8):
    """
    Train a YOLO model with specified configuration.
    
    Args:
        model_config: Path to model YAML config.
        pretrained: Path to pretrained weights.
        data_yaml: Path to data configuration.
        epochs: Number of training epochs.
        imgsz: Image size.
        batch: Batch size.
    """
    model = YOLO(model_config)
    model.load(pretrained)
    
    results = model.train(
        data=data_yaml,
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        patience=100,
        single_cls=True
    )
    
    return model, results

In [ ]:
# Train model (uncomment to execute)
# model, results = train_model(
#     model_config='yolo12x-obb.yaml',
#     pretrained='yolo12x.pt',
#     data_yaml='data.yaml',
#     epochs=300,
#     imgsz=1024,
#     batch=8
# )

In [ ]:
def evaluate_model(model_path: str, data_yaml: str = "data.yaml", 
                   split: str = "test", imgsz: int = 1024) -> Dict:
    """
    Evaluate a trained model and return metrics.
    """
    model = YOLO(model_path)
    metrics = model.val(
        data=data_yaml,
        split=split,
        save_json=True,
        save_txt=True,
        imgsz=imgsz,
        plots=True,
        batch=16,
        single_cls=True,
        save_conf=True
    )

    results = {
        'mAP50_95': metrics.box.map,
        'mAP50': metrics.box.map50,
        'mAP75': metrics.box.map75,
        'precision': metrics.box.p[0],
        'recall': metrics.box.r[0]
    }

    print("\n" + "="*50)
    print("Model Evaluation Results")
    print("="*50)
    print(f"mAP@.5:.95:   {results['mAP50_95']:.3f}")
    print(f"mAP@.5:       {results['mAP50']:.3f}")
    print(f"mAP@.75:      {results['mAP75']:.3f}")
    print(f"Precision:    {results['precision']:.3f}")
    print(f"Recall:       {results['recall']:.3f}")
    print("="*50)

    return results

In [ ]:
# Evaluate model (uncomment to execute)
# results = evaluate_model(
#     model_path='runs/obb/train/weights/best.pt',
#     data_yaml='data.yaml',
#     split='test'
# )

---
## Iteration Tracking

Template for tracking active learning iterations.

In [ ]:
# Create iteration tracking table
iteration_template = pd.DataFrame({
    'Iteration': [0, 1, 2, 3],
    'Training Samples': [200, 400, 600, 800],
    'Selection Strategy': ['Initial', 'Typical Diversity', 'Typical Diversity', 'Typical Diversity'],
    'mAP@.5': [0.0, 0.0, 0.0, 0.0],
    'mAP@.5:.95': [0.0, 0.0, 0.0, 0.0],
    'Precision': [0.0, 0.0, 0.0, 0.0],
    'Recall': [0.0, 0.0, 0.0, 0.0]
})

print("Active Learning Iteration Tracking Template:")
print(iteration_template.to_markdown(index=False))

---
## Summary

This notebook provides a complete active learning pipeline for object detection:

1. **Layer Analysis**: Identified optimal feature extraction layer
2. **Clustering**: Visualized data distribution and cluster structure
3. **Selection Strategies**: Compared multiple sampling approaches
4. **Training**: Iteratively improved model with selected samples

### Key Findings Template:
- Best feature extraction layer: Layer X
- Optimal clustering algorithm: K-Means / HDBSCAN
- Most effective selection strategy: [Strategy Name]
- Performance improvement: X% mAP increase per iteration